# E1 (IMDB) — Clean baseline
No poisoning. Reference point for CACC and the surrogate model used by E3's CBS selection.

Uses `EVAL_SIZE=5000` (subsample of the 25k test set) for speed -- swap to the full test set for your final publication numbers if time allows.

In [1]:
!pip install transformers datasets scikit-learn --quiet


In [6]:
import random, os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256
TARGET_LABEL = 1
POISON_RATE = 0.03   # single rate for ALL combos -- IMDB plateaus ~84-86% ASR from 0.03-0.1 regardless
                      # of higher rates or more epochs (confirmed), so 0.03 is the efficient operating point
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
EVAL_SIZE = 25000   # subsample of the 25k test set for speed; use full set for final publication numbers
EPOCHS = 3
print(DEVICE)

cuda


In [7]:
ds = load_dataset("stanfordnlp/imdb")
clean_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
full_test_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})
clean_valid_df = full_test_df.sample(n=EVAL_SIZE, random_state=SEED).reset_index(drop=True)
print("train:", clean_train_df.shape, "| eval subsample:", clean_valid_df.shape)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df, tok=None):
    tok = tok or tokenizer
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tok(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

train: (25000, 2) | eval subsample: (25000, 2)


## Train

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
train_ds = to_hf_dataset(clean_train_df)
valid_ds = to_hf_dataset(clean_valid_df)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

args = TrainingArguments(
    output_dir="./results_e1_clean_imdb", num_train_epochs=EPOCHS,
    per_device_train_batch_size=8, per_device_eval_batch_size=32,
    learning_rate=2e-5, eval_strategy="epoch", save_strategy="no",
    logging_steps=200, seed=SEED, report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds,
                   compute_metrics=compute_metrics)
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.287873,0.298783,0.906640,0.943311,0.865280,0.902612
2,0.198885,0.342568,0.919600,0.919734,0.919440,0.919587
3,0.099859,0.414084,0.919680,0.914114,0.926400,0.920216


TrainOutput(global_step=9375, training_loss=0.19754765146891276, metrics={'train_runtime': 2947.9203, 'train_samples_per_second': 25.442, 'train_steps_per_second': 3.18, 'total_flos': 9866664576000000.0, 'train_loss': 0.19754765146891276, 'epoch': 3.0})

## Evaluate + save
This model is reused as: (a) the E1 baseline, (b) the surrogate for E3's CBS scoring.

In [9]:
preds = np.argmax(trainer.predict(valid_ds).predictions, axis=-1)
cacc = accuracy_score(clean_valid_df["label"], preds)
p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], preds, average="binary")
e1_results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1}
print(e1_results)

model.save_pretrained("./models/e1_clean_imdb")
tokenizer.save_pretrained("./models/e1_clean_imdb")
print("saved e1_clean_imdb")

{'CACC': 0.91968, 'Precision': 0.9141143037574992, 'Recall': 0.9264, 'F1': 0.9202161474888748}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e1_clean_imdb


In [10]:
e1_results_df = pd.DataFrame(e1_results, index=[0])

In [12]:
e1_results_df.to_json("./results/e1_clean_imdb_results.json", orient="index", indent=4)